# Problem 1: Represent SHA-256 data in Python

## 32-bit word

A 32-bit word is a group of 32 binary digits (bits), treated as one unit by the algorithm. For example, `0x00000001` (represents decimal 1) and `0xFFFFFFFF` (represents decimal $2^{32}-1$) are both 32-bit words: each has eight hexadecimal digits, with each digit representing four bits.

FIPS 180-4 defines the SHA-256 algorithm in terms of 32-bit words.
A word contains exactly 32 bits, so its unsigned integer value is in the following range:

$$
0 \leq w \leq 2^{32} - 1.
$$

NumPy's `uint32` type represents values in this range directly. This fixed-width representation is useful because SHA-256 arithmetic is performed on 32-bit words; in particular, addition is modulo $2^{32}$ (FIPS 180-4, Sections 3.1 and 4.1.2). Python's built-in `int` has arbitrary precision, so using `uint32` makes the intended word width explicit.

In [6]:
import numpy as np

# Start with the largest value that fits in an unsigned 32-bit word.
word = np.uint32(0xFFFFFFFF)
# Represent 1 as a 32-bit word too, so both operands have the same width.
one = np.uint32(1)

# Force the result to remain uint32: overflow wraps modulo 2**32.
# ** means exponentiation in Python, so 2**32 is 2 raised to the power of 32.
wrapped = np.add(word, one, dtype=np.uint32)

# Convert to Python ints for formatting as eight-digit hexadecimal values.
print(f"word    = 0x{int(word):08X}")
print(f"word + 1 = 0x{int(wrapped):08X}")
# Check that the maximum 32-bit value plus 1 wraps around to zero.
assert wrapped == np.uint32(0)

word    = 0xFFFFFFFF
word + 1 = 0x00000000


## Sequences of 32-bit words

### What is a sequence?

Think of a word as one box. The box holds one number, stored as 32 on/off switches (bits). A sequence of words is just a row of those boxes. The boxes are numbered from $0$:

```text
position:     0          1          2          3
          +----------+----------+----------+----------+
          |   word   |   word   |   word   |   word   |
          +----------+----------+----------+----------+
```

SHA-256 does not hash one box at a time. It takes rows of boxes, mixes them together, and hands back a row of 8 boxes: the hash. So we need a way to store "a row of 32-bit boxes" in Python.

### How to store a row of boxes in Python

Use a NumPy arrays, always write `dtype=np.uint32`. Without it, NumPy picks bigger 64-bit boxes, and SHA-256 needs exactly 32. Example: `np.array([...], dtype=np.uint32)` means:

| Term | Plain meaning |
|:---|:---|
| `dtype` | "data type": what kind of thing every box hold. |
| `np.uint32` | The kind of box we want: a whole number made of 32 switches that can't be negative. u = unsigned, int = whole number, 32 = number of bits.|

In [4]:
import numpy as np

# A row of 4 boxes, each a 32-bit whole number.
row = np.array([10, 20, 30, 40], dtype=np.uint32)

print(row)        # all the boxes
print(row[0])     # the first box (position 0)
print(row[3])     # the last box (position 3)
print(row.size)   # how many boxes
print(row.shape)  # (4,) means: one row with 4 boxes
print(row.dtype)  # what each box holds

[10 20 30 40]
10
40
4
(4,)
uint32


### The four rows SHA-256 uses

FIPS 180-4 (Sections 3.1, 4.2.2, 5.2.1 and 6.2) says SHA-256 uses these four rows. Every box is 32 bits, so the size in bits is just the number of boxes $\times\ 32$.

| Row | What it is for | Boxes | Bits | Math name |
|:---|:---|:---:|:---:|:---:|
| Message block | Your message, cut into pieces. SHA-256 reads 512 bits at a time, so a long message is split into 512-bit pieces and each piece fills one row of 16 boxes. | 16 | 512 | $M_0, \dots, M_{15}$ |
| Message schedule | For each piece, SHA-256 stretches its 16 boxes into 64 boxes by mixing the earlier boxes together. The algorithm has 64 steps, and each step uses one of these boxes. | 64 | 2048 | $W_0, \dots, W_{63}$ |
| Hash value | The answer so far. It starts as 8 fixed numbers, is updated after every piece, and once the last piece is done these 8 boxes are the final hash ($8 \times 32 = 256$ bits). | 8 | 256 | $H_0, \dots, H_7$ |
| Round constants | A fixed list of 64 numbers, one for each of the 64 steps, copied from the standard. They are the same for every message and are mixed in at each step to scramble the numbers. | 64 | 2048 | $K_0, \dots, K_{63}$ |

Reading the math names: the letter names the row, the small number below it is the box position. So $W_5$ means "box 5 of the message schedule" and $H_0, \dots, H_7$ means "boxes 0, 1, 2, and so on up to 7 of the hash value". The standard also writes a small $(i)$ above the letter to say which block or hash value.

Hexadecimal: the standard writes each box as 8 hex digits, because each digit is 4 bits and $8 \times 4 = 32$.

The code below builds two of the rows:

- the message block for the text `"abc"` (Section 5.1.1)
- the starting hash value (Section 5.3.3)

In [5]:
import numpy as np

# Message block: 16 empty boxes (all zeros) to start with.
block = np.zeros(16, dtype=np.uint32)
# Box 0 holds the letters a, b, c (61 62 63) and then a single 1 bit (the 80).
block[0] = 0x61626380
# Box 15 holds the message length: 3 letters x 8 bits = 24 bits = 0x18.
block[15] = 0x00000018

# Starting hash value: 8 boxes, given in the standard.
H0 = np.array([
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
], dtype=np.uint32)

# Size in bits = number of boxes x 32.
print(f"block: {block.size} boxes = {block.size * 32} bits")
print(f"H0:    {H0.size} boxes = {H0.size * 32} bits")
assert block.dtype == np.uint32 and block.size * 32 == 512
assert H0.dtype == np.uint32 and H0.size * 32 == 256

# Print every box as 8 hex digits.
print("block =", " ".join(f"{int(w):08x}" for w in block))
print("H0    =", " ".join(f"{int(w):08x}" for w in H0))

block: 16 boxes = 512 bits
H0:    8 boxes = 256 bits
block = 61626380 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000018
H0    = 6a09e667 bb67ae85 3c6ef372 a54ff53a 510e527f 9b05688c 1f83d9ab 5be0cd19


## Input messages

## 512-bit message block

## Final 256-bit hash value

## Big-endian integers

# Problem 2: Implement SHA-256 bitwise operations

# Problem 3: Generate SHA-256 constants

# Problem 4: Implement SHA-256 padding and block parsing

# Problem 5: Implement the SHA-256 compression function

# Problem 6: Complete and verify SHA-256